# IgBLAST + abstar на двух problematic mouse sequences

Запускать из Jupyter в `bcr_env`. Cell 2 делает IgBLAST для 2 sequences, cell 3 делает abstar, cell 4 сравнивает side-by-side.


In [ ]:
# Запуск IgBLAST на problematic FASTA (2 sequences)
import os, subprocess, time
from pathlib import Path
import pandas as pd

BASE = Path("/data/user/epishkin/results/ERP003950")
FA = BASE / "problematic_sequences/input/problematic_mouse_2seq.fa"
IG_OUTDIR = BASE / "problematic_sequences/output/igblast"
IG_OUTDIR.mkdir(parents=True, exist_ok=True)
IG_OUT = IG_OUTDIR / "problematic_mouse_2seq_igblast.tsv"
IG_LOG = IG_OUTDIR / "problematic_mouse_2seq_igblast.log"

IGBLASTN = Path("/data/user/epishkin/igblast/bin/igblastn")
V = Path("/data/user/epishkin/results/internal_data/mouse/mouse_gl_V")
D = Path("/data/user/epishkin/results/internal_data/mouse/mouse_gl_D")
J = Path("/data/user/epishkin/results/internal_data/mouse/mouse_gl_J")
AUX = Path("/data/user/epishkin/results/optional_files/mouse_gl.aux")

os.environ["IGDATA"] = "/data/user/epishkin/igblast"
os.environ["PATH"] = "/data/user/epishkin/igblast/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/data/user/epishkin/conda/envs/bcr_env/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
for stale in [IG_OUT, IG_LOG]:
    if stale.exists():
        stale.unlink()

cmd = [str(IGBLASTN),
       "-germline_db_V", str(V),
       "-germline_db_D", str(D),
       "-germline_db_J", str(J),
       "-organism", "mouse",
       "-ig_seqtype", "Ig",
       "-domain_system", "imgt",
       "-query", str(FA),
       "-auxiliary_data", str(AUX),
       "-outfmt", "19",
       "-num_threads", "1",
       "-out", str(IG_OUT)]

print("Running IgBLAST on", FA, "->", IG_OUT)
t0 = time.time()
with IG_LOG.open("w") as log:
    rc = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, text=True, env=os.environ.copy()).returncode
print(f"rc={rc} elapsed={time.time() - t0:.0f}s size={IG_OUT.stat().st_size if IG_OUT.exists() else None}")
if rc != 0:
    print(IG_LOG.read_text(errors="ignore")[-2000:])
    raise RuntimeError(f"IgBLAST failed rc={rc}; see {IG_LOG}")

ig = pd.read_csv(IG_OUT, sep="\t", dtype=str)
cols = [c for c in ["sequence_id", "productive", "stop_codon", "v_call", "j_call", "cdr3_aa"] if c in ig.columns]
print(ig[cols].set_index("sequence_id").T)


In [ ]:
# Запуск abstar на problematic FASTA (2 sequences)
import os, subprocess, time, shutil
from pathlib import Path
import pandas as pd

BASE = Path("/data/user/epishkin/results/ERP003950")
FA = BASE / "problematic_sequences/input/problematic_mouse_2seq.fa"
OUT = BASE / "problematic_sequences/output/abstar"
OUT.mkdir(parents=True, exist_ok=True)
PROJ = OUT / "problematic_mouse_2seq_project"
if PROJ.exists():
    shutil.rmtree(str(PROJ))

ABSTAR = Path("/data/user/epishkin/conda/envs/bcr_env/bin/abstar")
if not ABSTAR.exists():
    raise FileNotFoundError(f"abstar binary not found: {ABSTAR}")

os.environ["PATH"] = "/data/user/epishkin/conda/envs/bcr_env/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/data/user/epishkin/conda/envs/bcr_env/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.chdir(str(OUT))
print("Running abstar on", FA, "->", PROJ)
t0 = time.time()
rc = subprocess.run(
    [str(ABSTAR), "run", str(FA), str(PROJ),
     "--germline_database", "c57bl6", "-o", "airr",
     "--n_processes", "2", "--verbose"],
    capture_output=True, text=True, env=os.environ.copy())
print(f"rc={rc.returncode} elapsed={time.time() - t0:.0f}s")
if rc.stdout:
    print("STDOUT tail:", rc.stdout[-1500:])
if rc.stderr:
    print("STDERR tail:", rc.stderr[-1500:])
if rc.returncode != 0:
    raise RuntimeError(f"abstar failed rc={rc.returncode}")

candidates = sorted(PROJ.rglob("*.tsv"), key=lambda p: p.stat().st_size, reverse=True)
if not candidates:
    raise RuntimeError(f"no abstar output TSV under {PROJ}")
dest = OUT / "problematic_mouse_2seq_abstar.tsv"
shutil.copy(str(candidates[0]), str(dest))
print("Saved:", dest)
ab = pd.read_csv(dest, sep="\t", dtype=str)
cols = [c for c in ["sequence_id", "productive", "stop_codon", "v_call", "j_call", "cdr3_aa"] if c in ab.columns]
print(ab[cols].set_index("sequence_id").T)


In [ ]:
# Side-by-side сравнение IgBLAST vs abstar
import pandas as pd
from pathlib import Path

BASE = Path("/data/user/epishkin/results/ERP003950")
IG = BASE / "problematic_sequences/output/igblast/problematic_mouse_2seq_igblast.tsv"
AB = BASE / "problematic_sequences/output/abstar/problematic_mouse_2seq_abstar.tsv"

ig = pd.read_csv(IG, sep="\t", dtype=str).set_index("sequence_id")
ab = pd.read_csv(AB, sep="\t", dtype=str).set_index("sequence_id")

rows = []
for sid in ["ko_seq_1", "ko_seq_2"]:
    r = {"sequence_id": sid}
    for prefix, df in [("igblast", ig), ("abstar", ab)]:
        if sid in df.index:
            for col in ["productive", "stop_codon", "v_call", "j_call", "cdr3_aa"]:
                if col in df.columns:
                    r[f"{prefix}_{col}"] = df.at[sid, col]
        else:
            r[f"{prefix}_found"] = False
    rows.append(r)

cmp_df = pd.DataFrame(rows)
print(cmp_df.T)
dest = BASE / "problematic_sequences/problematic_sequence_comparison_full.tsv"
cmp_df.to_csv(dest, sep="\t", index=False)
print("Written:", dest)
